# Global Air Pollution Analysis

## 1. Problem Definition
**Dataset:** Global Air Pollution Dataset from Kaggle.
**Topic:** Analyzing the distribution, correlation, and severity of different air pollutants (PM2.5, CO, Ozone, NO2) across various cities and countries globally.

**Analytical Questions:**
1. Are there significant geographical variations in overall air quality, and which countries experience the most severe PM2.5 pollution levels?
2. Which specific pollutant (CO, Ozone, NO2, or PM2.5) most strongly drives the overall Air Quality Index (AQI) globally?
3. How do the levels of different pollutants correlate with each other? Do highly polluted cities tend to suffer from combinations of dangerous pollutants simultaneously? 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv('global air pollution dataset.csv')

# Display the first few rows
df.head()

In [ ]:
# Check dataset dimensions, data types, and missing values
print("Dataset Shape:", df.shape)
print("\n--- Data Types & Non-Null Counts ---")
df.info()
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

In [ ]:
# Summary statistics for numerical columns
df.describe()

## 2. Data Understanding (Explanation of Findings)

**1. Data Shape:** 
Our underlying dataset has **23,463 rows** and **12 columns**. With over 23K rows, this satisfies the grading criteria requirement of having at least 1,000 rows and 6 columns.

**2. Data Types:** 
We have a mix of numerical and categorical variables. 
- *Categorical Data (`object` or `string`)*: `Country`, `City`, and the categorical AQI identifiers (`AQI Category`, `CO AQI Category`, `Ozone AQI Category`, etc.).
- *Numerical Data (`int64`)*: The actual continuous and discrete numerical variables representing the Air Quality measurements (e.g., `AQI Value`, `CO AQI Value`, `PM2.5 AQI Value`).

**3. Missing Values:** 
Out of 23,463 entries, we have a minor dataset completeness issue:
- `Country` has 427 missing values (~1.8%).
- `City` has 1 missing value.
All actual numerical measurement points and categorical values for the pollution indices show 0 missing values, meaning the data collected by devices is incredibly robust, but some geolocation matching algorithms failed to append the string identifiers.

**4. Summary Statistics:**
- *Overall AQI:* The mean Air Quality Index is about 72 (Moderate), but it spans all the way up to a heavy peak of 500 (Hazardous).
- *Pollutant Medians vs Means:* Looking closely at `PM2.5 AQI Value`, the standard deviation is massive (54.79) compared to the mean (68.51) and median (54.0), highlighting heavy skew. A few cities suffer from extreme particulate matter pollution (up to 500) while a vast majority stay near 35-55. 
- *Low Drivers:* The `CO AQI Value` and `NO2 AQI Value` mean metrics are exceptionally low compared to PM2.5, suggesting that PM2.5 and Ozone are likely the main drivers of hazardous ratings globally.

## 3. Data Cleaning

To ensure the integrity of our analysis, we will perform three meaningful data cleaning steps, accompanied by their justifications:

1. **Handling Missing Values:** We observed 427 missing `Country` entries and 1 missing `City` entry. Since these missing geographical identifiers represent a very small fraction of our dataset (~1.8%) and our questions rely on geographic grouping, we will drop these rows. Imputing dummy locations would distort spatial analysis.
2. **Removing Duplicate Rows:** Completely duplicated device readings can skew statistical distributions and correlations. We will strip all exact duplicate rows from the dataset to maintain unskewed data.
3. **Standardizing Column Names:** The original columns contain spaces and mixed casing (e.g., `PM2.5 AQI Value`). We will standardize these to lowercase `snake_case` (e.g., `pm2.5_aqi_value`) to ensure smooth programmatic access throughout the notebook workflow and avoid string parsing errors.

In [ ]:
# Creating a working copy for cleaning
df_clean = df.copy()

# Step 1: Handling Missing Values (Justification: missing geography prevents spatial analysis)
df_clean = df_clean.dropna(subset=['Country', 'City'])

# Step 2: Removing Duplicates (Justification: prevents skewed distributions)
df_clean = df_clean.drop_duplicates()

# Step 3: Standardizing Column Names (Justification: ensures programmatic syntax stability)
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(' ', '_')

# Checking shape and null values after cleaning
print("Original Shape:", df.shape)
print("Cleaned Shape:", df_clean.shape)
print("Duplicate Rows Remaining:", df_clean.duplicated().sum())
print("Missing Values Remaining:\n", df_clean.isnull().sum().sum())
print("New Column Names:\n", df_clean.columns.tolist())

## 4. Feature Engineering

We need to create at least two new derived columns to aid in our deeper analysis:

1. **`primary_pollutant`**: While the dataset provides an overall AQI value, it doesn't explicitly state which specific pollutant is driving that score in a single, clean categorical column. We will derive this by finding the maximum AQI value among the 4 individual pollutants (CO, Ozone, NO2, PM2.5) for every city.
2. **`hazard_severity_score`**: The `aqi_category` is a string (e.g., "Moderate", "Hazardous"). String-based categories limit our numerical operations and correlation modeling. We will map these categories to an ordinal scale (`1 = Good` to `6 = Hazardous`).

In [ ]:
# Feature 1: Primary Pollutant
# Find the column with the maximum AQI value among the 4 pollutant features for each row
pollutant_cols = ['co_aqi_value', 'ozone_aqi_value', 'no2_aqi_value', 'pm2.5_aqi_value']
df_clean['primary_pollutant'] = df_clean[pollutant_cols].idxmax(axis=1)

# Clean up the output string (e.g., 'pm2.5_aqi_value' -> 'PM2.5')
df_clean['primary_pollutant'] = df_clean['primary_pollutant'].str.replace('_aqi_value', '').str.upper()

# Feature 2: Hazard Severity Score
# Map the ordinal categorical string to a meaningful numerical score
severity_mapping = {
    'Good': 1,
    'Moderate': 2,
    'Unhealthy for Sensitive Groups': 3,
    'Unhealthy': 4,
    'Very Unhealthy': 5,
    'Hazardous': 6
}
df_clean['hazard_severity_score'] = df_clean['aqi_category'].map(severity_mapping)

# Check our new engineered columns
df_clean[['city', 'aqi_value', 'aqi_category', 'primary_pollutant', 'hazard_severity_score']].head(10)

## 5. Data Analysis

To thoroughly analyze the dataset, we will execute six distinct operations as required by the project specifications:
1. **Subgroup Comparison 1**: Average AQI score by `primary_pollutant`. Does PM2.5 or Ozone typically lead to higher overall AQI?
2. **Subgroup Comparison 2**: The top 10 countries with the worst average air quality.
3. **Relationship Analysis**: A correlation matrix between the four key pollutants (CO, Ozone, NO2, PM2.5).
4. **Outlier / Anomaly Analysis**: Identifying the most extremely polluted "Hazardous" outlier cities (AQI > 300).
5. **Custom NumPy Calculation**: Calculating the Coefficient of Variation ($CV = \frac{\sigma}{\mu}$) for each pollutant using numpy arrays to measure which pollutant has the most volatile/unpredictable spikes globally.
6. **Additional Metric (Frequency)**: Counting the global distribution of the new `hazard_severity_score` to see what tier most cities fall into.

In [ ]:
# 1. Subgroup Comparison 1: Mean AQI for each Primary Pollutant causing the rating
primary_pollutant_avg = df_clean.groupby('primary_pollutant')['aqi_value'].mean().sort_values(ascending=False).reset_index()
print("--- Average Total AQI driven by Primary Pollutant Type ---")
print(primary_pollutant_avg)

# 2. Subgroup Comparison 2: Top 10 Most Polluted Countries (by Mean AQI)
country_avg_aqi = df_clean.groupby('country')[['aqi_value', 'pm2.5_aqi_value']].mean().sort_values(by='aqi_value', ascending=False).head(10)
print("\n--- Top 10 Most Polluted Countries (Average AQI) ---")
print(country_avg_aqi)

# 3. Relationship Analysis: Correlation between specific pollutants
# Only correlate the core numerical variables of interest
corr_matrix = df_clean[['co_aqi_value', 'ozone_aqi_value', 'no2_aqi_value', 'pm2.5_aqi_value', 'aqi_value']].corr()
print("\n--- Biometric Correlation Matrix (Relationships) ---")
print(corr_matrix)

# 4. Outlier Analysis: Identifying Hazardous Outliers (AQI > 300)
# According to global standards, AQI over 300 is deemed "Hazardous" and a severe health emergency
hazardous_outliers = df_clean[df_clean['aqi_value'] > 300].sort_values(by='aqi_value', ascending=False)
print(f"\n--- Hazardous Anomaly Outliers ---")
print(f"Total cities reporting Hazardous AQI (>300): {len(hazardous_outliers)} out of {len(df_clean)} (~{round((len(hazardous_outliers)/len(df_clean))*100, 2)}%)")
print("Top 5 Worst Outliers:")
print(hazardous_outliers[['city', 'country', 'aqi_value', 'primary_pollutant']].head())

# 5. Custom NumPy Calculation: Coefficient of Variation (Volatility Analysis)
# Coefficient of Variation (CV) measures dispersion relative to the mean. A higher CV means spikes are much less predictable.
import numpy as np

pollutants = ['co_aqi_value', 'ozone_aqi_value', 'no2_aqi_value', 'pm2.5_aqi_value']
cv_results = {}

for pollutant in pollutants:
    pollutant_array = df_clean[pollutant].to_numpy()
    # Calculate CV: (Standard Deviation / Mean)
    cv = np.std(pollutant_array) / np.mean(pollutant_array)
    cv_results[pollutant] = cv

print("\n--- Custom NumPy Calculation: Coefficient of Variation (Volatility) ---")
for p, cv in cv_results.items():
    print(f"{p}: {round(cv, 3)}")

# 6. Additional Support Metric: Frequency Distribution of Global Severity Scores
severity_dist = df_clean['hazard_severity_score'].value_counts().sort_index()
print("\n--- Distribution of Cities per Hazard Severity Score (1=Good, 6=Hazardous) ---")
print(severity_dist)

## 6. Data Visualization

The following visual representations highlight the most pressing questions defined in the project scope using pure `matplotlib`. Every chart answers a specific behavior in global pollutant distributions.

In [ ]:
# Visualization 1: Top 10 Most Polluted Countries (Bar Chart)
fig, ax = plt.subplots(figsize=(10, 6))
country_names = country_avg_aqi.index
avg_aqi_scores = country_avg_aqi['aqi_value']

ax.barh(country_names[::-1], avg_aqi_scores[::-1], color='crimson', edgecolor='black')
ax.set_title('Top 10 Most Polluted Countries by Average AQI', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Air Quality Index (AQI)', fontsize=12)
ax.set_ylabel('Country', fontsize=12)
ax.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**Insight 1:** The horizontal bar chart highlights the 10 nations suffering from the most alarming long-term air pollution, with Bahrain topping the chart near an average AQI of 200 (Highly Unhealthy). The concentration of these nations largely spans industrializing or arid regions across the Middle East, South Asia, and North Africa. This validates our first analytical question regarding the steep geographical disparity in air quality metrics.

In [ ]:
# Visualization 2: Distribution of the Primary Pollutant (Pie Chart)
pollutant_counts = df_clean['primary_pollutant'].value_counts()
labels = pollutant_counts.index
sizes = pollutant_counts.values
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors,
       wedgeprops={'edgecolor': 'black', 'linewidth': 1})
ax.set_title('Global Distribution of the Primary Driving Pollutant', fontsize=14, fontweight='bold')
plt.show()

**Insight 2:** Our pie chart explicitly reveals what type of pollutant causes the worst air days in almost all measured cities. PM2.5 dominates by a monumental margin, accounting for nearly 50%, while Ozone heavily dictates the other 43%. This means that dealing with poor AQI largely requires filtering fine particulate matter and tropospheric ozone exclusively, rather than worrying about excessive Carbon Monoxide (which registers at 0%).

In [ ]:
# Visualization 3: Relationship between PM2.5 and Overall AQI (Scatter Plot)
fig, ax = plt.subplots(figsize=(10, 6))

x_data = df_clean['pm2.5_aqi_value']
y_data = df_clean['aqi_value']

ax.scatter(x_data, y_data, color='purple', alpha=0.5, edgecolor='none')
ax.set_title('Correlation Between PM2.5 and Overall AQI Score', fontsize=14, fontweight='bold')
ax.set_xlabel('PM2.5 AQI Share', fontsize=12)
ax.set_ylabel('Total City AQI Value', fontsize=12)
ax.grid(True, linestyle=':', alpha=0.6)

# Adding a trendline limit reference
ax.plot([0, 500], [0, 500], color='black', linestyle='--')

plt.tight_layout()
plt.show()

**Insight 3:** This scatter plot plots individual city PM2.5 levels against their respective total AQI. The massive, thick purple line trailing directly along the black x=y line confirms our coefficient correlation analysis: PM2.5 almost entirely dictates max daily spikes for high-scoring cities. The points floating above the line (where Total AQI is higher than the PM2.5 share) indicate cities where Ozone is dragging the base limit up independently of PM.

In [ ]:
# Visualization 4: Distribution Histogram of Worldwide Air Quality
fig, ax = plt.subplots(figsize=(10, 6))

hist_data = df_clean['aqi_value']
ax.hist(hist_data, bins=50, color='teal', edgecolor='black', alpha=0.7)

ax.set_title('Global Distribution Curve of Daily Mean AQI Values', fontsize=14, fontweight='bold')
ax.set_xlabel('Air Quality Index', fontsize=12)
ax.set_ylabel('Number of Cities', fontsize=12)

# Danger Zone threshold lines
ax.axvline(x=100, color='orange', linestyle='--', linewidth=2, label='Unhealthy (100+)')
ax.axvline(x=300, color='red', linestyle='--', linewidth=2, label='Hazardous (300+)')

ax.legend()
plt.tight_layout()
plt.show()

**Insight 4:** The global histogram outlines a massive right-skewed distribution. Positively, the enormous peak on the far left reveals that the majority of cities hover near 50-70 (Moderate/Good). However, the long right tail bleeding past the 100 threshold line and spiking occasionally past the 300 Hazardous threshold line tells us that bad pollution is highly localized, severely afflicting specific anomalous zones rather than being a steadily worsening global average.